# MGT5493 — Lab 4 Homework
## Logic, iteration, functions, and conditional classification

**100 marks — 90 automatic and 10 manual.**

Use the supplied `product_performance.csv` and require the installation of `Homework_marking_system` in the notebook working directory.

Save the named outputs requested below. Complete answer cells and keep the grading cells. Restart the kernel and Run All before reviewing feedback.

Thresholds include equality only where the instructions use >=.

Sales equal to zero give an undefined profit margin (`NaN`).

A margin-dependent classification of an undefined margin must be `Unknown`.

Keep the original `df` and work with `df_work` for derived columns.

The questions assess NumPy conditions, for loops, if/elif/else, and apply from the demonstration.

Automatic tests check results and function behavior. The final manual review also checks the required Python constructs.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from lab4_marking import create_marking

marking = create_marking("product_performance.csv")

## Q1. Load and inspect (5 marks)

1. Read `product_performance.csv` into `df` without changing it.
2. Save `data_shape = df.shape`
3. the first five rows as `first_5`
4. numeric descriptive statistics as `data_statistics`

Display them.

In [2]:
### BEGIN SOLUTION
df = pd.read_csv("product_performance.csv")
data_shape = df.shape
first_5 = df.head().copy()
data_statistics = df.describe()
print(data_shape)
display(first_5, data_statistics)
### END SOLUTION

(400, 5)


,Product,Category,Sales,Cost,Profit
0,FAS_014,FAS,387.14,284.85,102.29
1,FAS_064,FAS,610.10,490.24,119.86
2,OFF_074,OFF,763.30,584.19,179.11
3,HOM_038,HOM,488.70,387.14,101.56
4,OFF_040,OFF,425.76,372.19,53.56


,Sales,Cost,Profit
count,400.000000,400.000000,400.000000
mean,504.038175,405.369975,98.668025
std,230.512196,204.706113,114.227750
min,0.000000,0.000000,-254.210000
25%,352.980000,257.557500,24.677500
50%,510.445000,401.205000,88.005000
75%,653.105000,524.110000,159.240000
max,1199.570000,1133.630000,479.480000


In [3]:
marking.check('q1-data', df=df)

✅ q1-data: 2/2 marks


In [4]:
marking.check('q1-shape', data_shape=data_shape)

✅ q1-shape: 1/1 marks


In [5]:
marking.check('q1-head', first_5=first_5)

✅ q1-head: 1/1 marks


In [6]:
marking.check('q1-statistics', data_statistics=data_statistics)

✅ q1-statistics: 1/1 marks


## Q2. Profit margins and numeric data (10 marks)

1. Create `df_work = df.copy()`. Add only `Profit_margin` to it, calculated as Profit / Sales.
2. Replace a zero denominator with NaN before division
3. Create `df_num` with columns `Sales`, `Cost`, `Profit`, `Profit_margin` in that order

Keep row order and row index, and display the first five rows.

In [7]:
### BEGIN SOLUTION
df_work = df.copy()
df_work["Profit_margin"] = df_work["Profit"] / df_work["Sales"].replace(0, np.nan)
df_num = df_work[["Sales", "Cost", "Profit", "Profit_margin"]].copy()
display(df_num.head())
### END SOLUTION

,Sales,Cost,Profit,Profit_margin
0,387.14,284.85,102.29,0.264220
1,610.10,490.24,119.86,0.196460
2,763.30,584.19,179.11,0.234652
3,488.70,387.14,101.56,0.207817
4,425.76,372.19,53.56,0.125799


In [8]:
marking.check('q2-margins', df_work=df_work)

✅ q2-margins: 6/6 marks


In [9]:
marking.check('q2-numeric', df_num=df_num)

✅ q2-numeric: 4/4 marks


## Q3. Two-way profit classification (10 marks)

Use NumPy conditional selection to create the Series `profit_flags`.
- Unknown when Profit_margin is undefined.
- High when Profit_margin >= 0.25.
- Low otherwise.

Save `profit_counts` as a Series indexed High, Low, Unknown in that order.

Include a zero count for any absent class. Keep the same row index as df_work.

Store outputs separately so df_work still has the columns requested in Q2.

In [16]:
### BEGIN SOLUTION
margin = df_work["Profit_margin"]
profit_flags = pd.Series(
    np.where(margin.isna(), "Unknown", np.where(margin >= 0.25, "High", "Low")),
    index=df_work.index,
)
profit_counts = profit_flags.value_counts().reindex(['High', 'Low', 'Unknown'], fill_value=0)
display(profit_flags, profit_counts)
### END SOLUTION

0         High
1          Low
2          Low
3          Low
4          Low
        ...   
395        Low
396        Low
397        Low
398    Unknown
399       High
Length: 400, dtype: str

High       162
Low        232
Unknown      6
Name: count, dtype: int64

In [14]:
marking.check('q3-labels', profit_flags=profit_flags)

✅ q3-labels: 6/6 marks


In [17]:
marking.check('q3-counts', profit_counts=profit_counts)

✅ q3-counts: 4/4 marks


## Q4. Sales relative to the median (10 marks)

Calculate `median_sales` using all Sales values.

Use NumPy conditional selection to create `sales_levels`: Above_med for Sales >= median_sales and Below_med otherwise

Save `sales_counts` indexed Above_med, Below_med in that order, including zero counts if needed (3).

Keep the original row index in sales_levels. Display the results.

In [24]:
### BEGIN SOLUTION
median_sales = df_work["Sales"].median()
sales_levels = pd.Series(
    np.where(df_work["Sales"] >= median_sales, "Above_med", "Below_med"),
    index=df_work.index,
)

sales_counts = sales_levels.value_counts().reindex(["Above_med", "Below_med"], fill_value=0)
print(median_sales)
display(sales_counts)
### END SOLUTION

510.445


Above_med    200
Below_med    200
Name: count, dtype: int64

In [20]:
marking.check('q4-median', median_sales=median_sales)

✅ q4-median: 2/2 marks


In [21]:
marking.check('q4-labels', sales_levels=sales_levels)

✅ q4-labels: 5/5 marks


In [22]:
marking.check('q4-counts', sales_counts=sales_counts)

✅ q4-counts: 3/3 marks


## Q5. Category iteration (15 marks)

Use a `for` loop over the unique Category values.
For each category, filter its rows and collect:
- The number of rows in the dictionary `category_counts`, keyed by category.
- Mean Profit in the dictionary `category_avg_profit`, keyed by category.

A product count means the number of rows. Calculate mean Profit, rather than mean margin.
Use looping and filtering for this question. Your loop implementation will also be reviewed in Q9.

In [28]:
### BEGIN SOLUTION
category_counts = {}
category_avg_profit = {}

for category in df_work["Category"].unique():
    subset = df_work[df_work["Category"] == category]
    category_counts[category] = len(subset)
    category_avg_profit[category] = subset["Profit"].mean()
print(category_counts)
print(category_avg_profit)
### END SOLUTION

{'FAS': 89, 'OFF': 95, 'HOM': 118, 'ELE': 98}
{'FAS': np.float64(116.43797752808989), 'OFF': np.float64(94.68463157894742), 'HOM': np.float64(84.85254237288135), 'ELE': np.float64(103.02642857142857)}


In [26]:
marking.check('q5-counts', category_counts=category_counts)

✅ q5-counts: 7/7 marks


In [27]:
marking.check('q5-means', category_avg_profit=category_avg_profit)

✅ q5-means: 8/8 marks


## Q6. Functions and apply (15 marks)

1. Define `margin_check(margin, threshold=0.25)` using Python conditional statements.

Return Unknown for a missing margin, High for margin >= threshold, and Low otherwise.
Apply it to Profit_margin and save the resulting Series as `margin_labels`.

2. Define `profit_category(margin)` using if/elif/else.

Return Unknown for a missing margin, Very_high for margin >= 0.30,
Medium for 0.15 <= margin < 0.30, and Low otherwise.
Apply it and save `profit_categories`. Preserve the row indices.

Functions must work on new values and changed thresholds, not just the dataset rows.

In [29]:
### BEGIN SOLUTION
def margin_check(margin, threshold=0.25):
    if pd.isna(margin):
        return "Unknown"
    if margin >= threshold:
        return "High"
    return "Low"

def profit_category(margin):
    if pd.isna(margin):
        return "Unknown"
    elif margin >= 0.30:
        return "Very_high"
    elif margin >= 0.15:
        return "Medium"
    else:
        return "Low"

margin_labels = df_work["Profit_margin"].apply(margin_check)
profit_categories = df_work["Profit_margin"].apply(profit_category)
display(margin_labels, profit_categories)
### END SOLUTION

0         High
1          Low
2          Low
3          Low
4          Low
        ...   
395        Low
396        Low
397        Low
398    Unknown
399       High
Name: Profit_margin, Length: 400, dtype: str

0         Medium
1         Medium
2         Medium
3         Medium
4            Low
         ...    
395          Low
396       Medium
397       Medium
398      Unknown
399    Very_high
Name: Profit_margin, Length: 400, dtype: str

In [30]:
marking.check('q6-margin-function', margin_check=margin_check)

✅ q6-margin-function: 4/4 marks


In [31]:
marking.check('q6-margin-apply', margin_labels=margin_labels)

✅ q6-margin-apply: 3/3 marks


In [32]:
marking.check('q6-category-function', profit_category=profit_category)

✅ q6-category-function: 5/5 marks


In [33]:
marking.check('q6-category-apply', profit_categories=profit_categories)

✅ q6-category-apply: 3/3 marks


## Q7. Promotion rules and a summary table (10 marks)

Create `promotion_flags` as a Series containing Consider_Promotion when Profit >= 80 and No_Need otherwise

Preserve row indices. Use NumPy conditional selection.

Create `promotion_summary_answer`, a table of row counts by Category and promotion label

Use Category as the index, sorted alphabetically, and columns Consider_Promotion, No_Need in that order.
Fill absent combinations with zero. You may use groupby to build this summary.

*Hint: .assign() function for creating a view; and .unstack() could fill absent combiations with zero*

In [34]:
### BEGIN SOLUTION
promotion_flags = pd.Series(
    np.where(df_work["Profit"] >= 80, "Consider_Promotion", "No_Need"),
    index=df_work.index,
)
promotion_summary_answer = (
    df_work.assign(Promotion=promotion_flags)
    .groupby(["Category", "Promotion"]).size()
    .unstack(fill_value=0)
    .reindex(columns=["Consider_Promotion", "No_Need"], fill_value=0)
    .sort_index()
)
display(promotion_summary_answer)
### END SOLUTION

Promotion,Consider_Promotion,No_Need
Category,,
ELE,53,45
FAS,57,32
HOM,51,67
OFF,48,47


In [35]:
marking.check('q7-flags', promotion_flags=promotion_flags)

✅ q7-flags: 5/5 marks


In [36]:
marking.check('q7-summary', promotion_summary_answer=promotion_summary_answer)

✅ q7-summary: 5/5 marks


## Q8. Combined conditions and row-wise apply (15 marks)

Define `risk_flag(row, sales_benchmark)` using conditional statements (8).
The row contains Sales and Profit_margin.
- Return Unknown if Profit_margin is undefined.
- Return High_Risk if Profit_margin < 0.15 AND Sales > sales_benchmark.
- Return Low_Risk otherwise.

Use `df_work.apply(..., axis=1)` with median_sales as the benchmark.
Save the Series as `risk_labels` and retain row indices (7).
The benchmark parameter must work with other input values too.

*Hint: .apply(lambda...:) could help for function have more than one input*

In [38]:
### BEGIN SOLUTION
def risk_flag(row, sales_benchmark):
    if pd.isna(row["Profit_margin"]):
        return "Unknown"
    if row["Profit_margin"] < 0.15 and row["Sales"] > sales_benchmark:
        return "High_Risk"
    return "Low_Risk"

risk_labels = df_work.apply(lambda row: risk_flag(row, median_sales), axis=1)
display(risk_labels.value_counts())
### END SOLUTION

Low_Risk     323
High_Risk     71
Unknown        6
Name: count, dtype: int64

In [39]:
marking.check('q8-function', risk_flag=risk_flag)

✅ q8-function: 8/8 marks


In [40]:
marking.check('q8-apply', risk_labels=risk_labels)

✅ q8-apply: 7/7 marks


## Q9. Code review and reflection (10 manual marks)

The marker will review the code and your response below.
- Q5 uses a for loop, per-category filtering, and collection into dictionaries (3).
- Q3/Q4/Q7 use NumPy selection, and Q6/Q8 use Python conditionals and the requested apply methods (4).
- Your written response explains why Sales == 0 requires care, why equality at a threshold matters,
  and one trade-off between loops and vectorized operations (1 mark for each, 3 total).

Profit / Sales is undefined for zero Sales, so I keep NaN and use Unknown for margin-dependent labels.
Using >= includes a value exactly at a threshold. For risk, Sales must be strictly greater than the benchmark,
so a value equal to it remains Low_Risk when the margin is defined.
Loops let me inspect and customise each category calculation, while vectorized operations generally express
whole-column rules more concisely and often run faster on larger datasets.

In [41]:
marking.summarise_marks()

q1-data: passed (2 marks available)
q1-shape: passed (1 marks available)
q1-head: passed (1 marks available)
q1-statistics: passed (1 marks available)
q2-margins: passed (6 marks available)
q2-numeric: passed (4 marks available)
q3-labels: passed (6 marks available)
q3-counts: passed (4 marks available)
q4-median: passed (2 marks available)
q4-labels: passed (5 marks available)
q4-counts: passed (3 marks available)
q5-counts: passed (7 marks available)
q5-means: passed (8 marks available)
q6-margin-function: passed (4 marks available)
q6-margin-apply: passed (3 marks available)
q6-category-function: passed (5 marks available)
q6-category-apply: passed (3 marks available)
q7-flags: passed (5 marks available)
q7-summary: passed (5 marks available)
q8-function: passed (8 marks available)
q8-apply: passed (7 marks available)
Visible checks: 90/90
✅ All 21 visible tests passed.
Hidden marks: 0; manual marks: 10.
Provisional: restart the kernel and Run All to check current answers.


{'earned': 90,
 'visible_total': 90,
 'hidden_total': 0,
 'manual_total': 10,
 'statuses': {'q1-data': 'passed',
  'q1-shape': 'passed',
  'q1-head': 'passed',
  'q1-statistics': 'passed',
  'q2-margins': 'passed',
  'q2-numeric': 'passed',
  'q3-labels': 'passed',
  'q3-counts': 'passed',
  'q4-median': 'passed',
  'q4-labels': 'passed',
  'q4-counts': 'passed',
  'q5-counts': 'passed',
  'q5-means': 'passed',
  'q6-margin-function': 'passed',
  'q6-margin-apply': 'passed',
  'q6-category-function': 'passed',
  'q6-category-apply': 'passed',
  'q7-flags': 'passed',
  'q7-summary': 'passed',
  'q8-function': 'passed',
  'q8-apply': 'passed'},
 'all_passed': True}